In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
plt.rcParams.update(plt.rcParamsDefault)

# 커스텀 팔레트
custom_cmap = sns.blend_palette(['#002D72', '#f1f0ec','#ffb659'], as_cmap=True)

# 컬러맵 프리뷰용
data = np.random.randn(10, 10)
sns.heatmap(data, cmap=custom_cmap)
plt.show()

# 그래프 기본 테마 설정
sns.set_theme(style="whitegrid", font_scale=1) # 블루톤

# 그리드 색상 조절
plt.rcParams.update({
    "grid.color": ".8",          # 그리드 색상
    "grid.linestyle": "--",       # 그리드 점선 스타일
    "grid.linewidth": 0.8,        # 그리드 두께
    "axes.grid": True,            # 그리드 항상 켜기
    "axes.edgecolor": ".8",       # 축 테두리 색상
})

# 막대그래프 관련 설정
plt.rcParams.update({
    "lines.linewidth": 2,
    "lines.marker": "D",          # 전역 마커 설정
    "lines.markersize": 7        # 마커 크기
})

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 나눔스퀘어
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 15, 9
plt.rcParams['axes.titlesize'] = 16 # 제목 폰트 사이즈
plt.rcParams['axes.labelsize'] = 14 # 라벨 폰트 사이즈
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.fontset'] = 'cm'

In [ ]:
# 알아서 째라...
def get_palette(n):
    return [custom_cmap(i / (n - 1)) for i in range(n)]

# 노동비용, 법정노동비용, 법정외노동비용
- 사대보험은 전자입니다.

In [ ]:
# 노동비용
nodong_total = pd.read_csv('data/nodong_total.csv')
law_nodong_won = pd.read_csv('data/law_nodong_total_won.csv')
law_nodong_rate = pd.read_csv('data/law_nodong_total_rate.csv')
notlaw_nodong_total = pd.read_csv('data/notlaw_nodong_total.csv')

In [ ]:
nodong_total

In [ ]:
law_nodong_rate

In [ ]:
law_nodong_won

In [ ]:
notlaw_nodong_total

## 전체 빼고 갑니다

In [ ]:
nodong_total = nodong_total.query('산업분류 != "전체"')
law_nodong_rate = law_nodong_rate.query('산업분류 != "전체"')
law_nodong_won = law_nodong_won.query('산업분류 != "전체"')
notlaw_nodong_total = notlaw_nodong_total.query('산업분류 != "전체"')

# 노동비용
## 최근 6개년 추이 (전체)

In [ ]:
direct_list = ['직접노동비용(계)', '정액 및 초과급여', '상여금 및 성과금']
nodong_direct = nodong_total.query('항목 in @direct_list')

In [ ]:
indirect_list = ['간접노동비용(계)','퇴직급여 등의 비용','법정노동비용','법정외 복지비용','채용관련비용(모집비)', '교육훈련비용']
nodong_indirect = nodong_total.query('항목 in @indirect_list')


### 직접노동비용

In [ ]:
# 픽
nodong_direct_total = nodong_direct.query('항목 == "직접노동비용(계)"')

# 시각화
sns.lineplot(nodong_direct_total, x = '연도', y = '비용(만원)', hue = '산업분류', palette='tab20')
plt.title('최근 6개년 직접노동비용 추이', y = 1.01)
plt.xlabel('연도')
plt.ylabel('직접노동비용 (만원)')
plt.ticklabel_format(axis='y', style='plain')
plt.legend(bbox_to_anchor=(1, 1))
plt.xticks([2019, 2020, 2021, 2022, 2023, 2024]) # 얘는 왜 이걸 해줘야 년도로 나올까...
plt.show()

### 간접노동비용

In [ ]:
# 픽
nodong_indirect_total = nodong_indirect.query('항목 == "간접노동비용(계)"')

# 시각화
sns.lineplot(nodong_indirect_total, x = '연도', y = '비용(만원)', hue = '산업분류', palette='tab20')
plt.title('최근 6개년 간접노동비용 추이', y = 1.01)
plt.xlabel('연도')
plt.ylabel('간접노동비용 (만원)')
plt.ticklabel_format(axis='y', style='plain')
plt.legend(bbox_to_anchor=(1, 1))
plt.xticks([2019, 2020, 2021, 2022, 2023, 2024]) # 얘는 왜 이걸 해줘야 년도로 나올까...
plt.show()

- 직접노동비용의 경우 좀 꺾이는 부분은 있어도 해가 갈수록 상승하는 추세인 듯 하군요.

## 최근 6개년 평균 (전채)

### 직접노동비용

In [ ]:
nodong_direct_mean = nodong_direct_total.groupby(['산업분류','항목']).agg({'비용(만원)':'mean'}).reset_index()
nodong_direct_mean = nodong_direct_mean.sort_values('비용(만원)',ascending=False)
nodong_direct_mean

In [ ]:
ax = sns.barplot(nodong_direct_mean, x = '산업분류', y = '비용(만원)', hue = '산업분류', palette=get_palette(nodong_direct_mean['산업분류'].nunique()))
for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%d', padding=3, fontsize=10)

plt.title('업종별 최근 6개년 직접노동비용 총계 평균', y = 1.01)
plt.xlabel('업종')
plt.ylabel('직접노동비용 평균 (만원)')
plt.axhline(np.median(nodong_direct_mean['비용(만원)']), linestyle='dashed', color='#cc0000', alpha=0.5) # 중앙값
plt.axhline(np.mean(nodong_direct_mean['비용(만원)']), linestyle='dashed', color='#ccaa00', alpha=0.5) # 평균
plt.ticklabel_format(axis='y', style='plain')
plt.xticks(rotation=90)
plt.show()

- 제조업은 그렇게 많이 나가진 않네요? 

### 간접노동비용

In [ ]:
nodong_indirect_mean = nodong_indirect_total.groupby(['산업분류','항목']).agg({'비용(만원)':'mean'}).reset_index()
nodong_indirect_mean = nodong_indirect_mean.sort_values('비용(만원)',ascending=False)
nodong_indirect_mean

In [ ]:
ax = sns.barplot(nodong_indirect_mean, x = '산업분류', y = '비용(만원)', hue = '산업분류', palette=get_palette(nodong_indirect_mean['산업분류'].nunique()))
for container in ax.containers:
    # fmt='%d'는 정수로 표시, label_type='edge'는 막대 끝에 표시
    ax.bar_label(container, fmt='%d', padding=3, fontsize=10)

plt.title('업종별 최근 6개년 간접노동비용 총계 평균', y = 1.01)
plt.xlabel('업종')
plt.ylabel('간접노동비용 평균 (만원)')
plt.axhline(np.median(nodong_indirect_mean['비용(만원)']), linestyle='dashed', color='#cc0000', alpha=0.5) # 중앙값
plt.axhline(np.mean(nodong_indirect_mean['비용(만원)']), linestyle='dashed', color='#ccaa00', alpha=0.5) # 평균
plt.ticklabel_format(axis='y', style='plain')
plt.xticks(rotation=90)
plt.show()

# 법적노동비용

In [ ]:
law_nodong_won = law_nodong_won.query('산업분류 != "전체"')
law_nodong_won

In [ ]:
law_nodong_rate = law_nodong_rate.query('산업분류 != "전체"')

## 법정노동비용 추이

In [ ]:
# 픽
nodong_law_total = law_nodong_won.query('지표 == "법정노동비용(계)"')

# 시각화
sns.lineplot(nodong_law_total, x = '연도', y = '비용(만원)', hue = '산업분류', palette='tab20')
plt.title('최근 6개년 법정노동비용 추이', y = 1.01)
plt.xlabel('연도')
plt.ylabel('법정노동비용 (만원)')
plt.ticklabel_format(axis='y', style='plain')
plt.legend(bbox_to_anchor=(1, 1))
plt.xticks([2019, 2020, 2021, 2022, 2023, 2024]) # 얘는 왜 이걸 해줘야 년도로 나올까...
plt.show()

## 4대보험 음...
- 아 이거 안내면요? 노동부가 이놈합니다. 

In [ ]:
law_nodong_won['지표'].value_counts()

In [ ]:
law_nodong_rate['지표'].value_counts()

In [ ]:
# 4대보험: 건보료 산재보험료 국민연금 고용보험료
# 그 급여명세서 보시면 알아서 월급에서 나가는 그거 있어요
health_insurance = law_nodong_won.query('지표 == "건강보험료"') # 건보료
accident_insurance = law_nodong_won.query('지표 == "산재보험료"') # 산재보험료
nps = law_nodong_won.query('지표 == "국민연금"') # 그거 그거 회사 때려치면 내라고 우편 오는거
employee_insurance = law_nodong_won.query('지표 == "고용보험료"') # 건보료

### 전업종(각개)

In [ ]:
# 시각화
sns.lineplot(health_insurance, x = '연도', y = '비용(만원)', hue = '산업분류', palette='tab20')
plt.title('최근 6개년 건강보험료 추이', y = 1.01)
plt.xlabel('연도')
plt.ylabel('건강보험료 (만원)')
plt.ticklabel_format(axis='y', style='plain')
plt.legend(bbox_to_anchor=(1, 1))
plt.xticks([2019, 2020, 2021, 2022, 2023, 2024]) # 얘는 왜 이걸 해줘야 년도로 나올까...
plt.show()

In [ ]:
# 시각화
sns.lineplot(accident_insurance, x = '연도', y = '비용(만원)', hue = '산업분류', palette='tab20')
plt.title('최근 6개년 산재보험료 추이', y = 1.01)
plt.xlabel('연도')
plt.ylabel('산재보험료 (만원)')
plt.ticklabel_format(axis='y', style='plain')
plt.legend(bbox_to_anchor=(1, 1))
plt.xticks([2019, 2020, 2021, 2022, 2023, 2024]) # 얘는 왜 이걸 해줘야 년도로 나올까...
plt.show()

### 제조업만(4 in 1)